# Sequence-модель: окно 30–60 баров

**Шаг 4 экспериментов.** Окна 30 и 60 для учёта истории. Rolling stats по окну вместо LSTM (избегаем OOM).

## 1. Импорты и подготовка

In [ ]:
import sys, os, numpy as np, pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
import joblib

_root = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) in ('01_data_prep','02_targets','03_features','04_models','05_experiments') else os.getcwd()
if _root not in sys.path:
    sys.path.insert(0, _root)
import lightgbm as lgb

labeled_path = os.path.join(_root, 'outputs', 'data_labeled_tp_sl_1_05.parquet')
feat_path = os.path.join(_root, 'outputs', 'features_selected_tp_sl_1_05.txt')
if not os.path.exists(labeled_path):
    raise FileNotFoundError('Запустите 04_Data_Labeling_And_Feature_Loading.ipynb')
if not os.path.exists(feat_path):
    raise FileNotFoundError('Запустите 06_Feature_Selection.ipynb')

df = pd.read_parquet(labeled_path)
with open(feat_path, encoding='utf-8') as f:
    BASE_FEATURES = [l.strip() for l in f if l.strip()]
BASE_FEATURES = [c for c in BASE_FEATURES if c in df.columns]

print(f'Строк: {len(df):,}, target=tp_sl_1_05 (колонка target)')

## 2. Rolling-фичи (окна 30, 60)

In [ ]:
key_feats = BASE_FEATURES[:10]
for w in [30, 60]:
    grp = df.groupby('session_key', group_keys=False)
    for c in key_feats:
        if c in df.columns:
            df[f'{c}_roll{w}_mean'] = grp[c].transform(lambda x: x.rolling(w, min_periods=1).mean())
            df[f'{c}_roll{w}_std'] = grp[c].transform(lambda x: x.rolling(w, min_periods=1).std().fillna(0))

SEQ_COLS = [c for c in df.columns if '_roll' in c]
FEATURES_SEQ = BASE_FEATURES + SEQ_COLS
FEATURES_SEQ = [c for c in FEATURES_SEQ if c in df.columns]
print(f'Фичей: baseline {len(BASE_FEATURES)}, +rolling {len(FEATURES_SEQ)}')

## 3. Обучение и бектест

In [ ]:
valid = df.dropna(subset=FEATURES_SEQ + ['target']).copy()
valid = valid[valid['target'].isin([-1.0, 1.0])]
valid['y'] = (valid['target'] == 1).astype(int)
valid['ret_next'] = valid.groupby('session_key')['close_price'].pct_change().shift(-1)

sessions = valid['session_key'].unique().tolist()
np.random.seed(42)
np.random.shuffle(sessions)
train_s = set(sessions[:int(0.7*len(sessions))])
test_s = set(sessions[int(0.85*len(sessions)):])
train_df = valid[valid['session_key'].isin(train_s)]
test_df = valid[valid['session_key'].isin(test_s)]

def run(features, comm=0.001):
    scaler = StandardScaler()
    X_train = scaler.fit_transform(train_df[features].fillna(0))
    X_test = scaler.transform(test_df[features].fillna(0))
    model = lgb.LGBMClassifier(n_estimators=100, max_depth=5, random_state=42, verbose=-1)
    model.fit(X_train, train_df['y'].values)
    auc = roc_auc_score(test_df['y'].values, model.predict_proba(X_test)[:, 1])
    bt = test_df.dropna(subset=['ret_next'])
    X_bt = scaler.transform(bt[features].fillna(0))
    p = model.predict_proba(X_bt)[:, 1]
    trade = np.where(p >= 0.6, 1, np.where(p <= 0.4, -1, 0))
    pnl = (trade * bt['ret_next'].values).sum() * 100 - (trade != 0).sum() * comm * 100
    return auc, pnl, (trade != 0).sum()

auc_b, pnl_b, tb = run(BASE_FEATURES)
auc_s, pnl_s, ts = run(FEATURES_SEQ)
print('Baseline: AUC=', f'{auc_b:.4f}', 'net PnL=', f'{pnl_b:.2f}%', 'trades=', tb)
print('Sequence(30,60): AUC=', f'{auc_s:.4f}', 'net PnL=', f'{pnl_s:.2f}%', 'trades=', ts)

## Итог шага 12

- Rolling mean/std по окнам 30 и 60.
- Сравнение AUC и net PnL: baseline vs sequence-фичи.